# [ALT] MedGemma-4B QLoRA 파인튜닝 — 확장 데이터셋 실험

## ⚠️ 이건 "메인"과 별개인 대안 학습 노트북입니다

`scripts/main_train_llm_lora.ipynb`(메인, KorMedMCQA+GenMedGPT-5k-ko)는 그대로 두고,
여기서는 Asan-AMC-Healthinfo / MedQA / KoMedInstruct-52k / snuh-ClinicalQA(의료 지식)와
ChainofDiagnosis-Ko / MedQA-Evol-Korean(추론 보강)을 추가/대체 조합으로 실험한다.
결과가 메인보다 나으면 그때 메인을 이걸로 교체하는 방식으로 간다 — 기존 어댑터
(`-v1`/`-v2`/`-main`)는 건드리지 않는다.

## Google Colab 무료 T4 환경용

`google/medgemma-4b-it`를 QLoRA 방식으로 파인튜닝하는 Notebook입니다.

### 주요 구성

* MedGemma 4B Instruct
* 4bit 양자화
* LoRA / QLoRA
* Google Colab T4 GPU
* 확장 데이터셋 조합 (11~13번대에서 구성)
* Google Drive 체크포인트 저장
* Hugging Face Hub LoRA Adapter 업로드 (`-alt` 접미사로 v1/v2/main과 구분)

### 실행 순서

반드시 아래 순서대로 실행하세요.

1. GPU 확인
2. 패키지 설치
3. 런타임 재시작
4. 환경 확인
5. Hugging Face 로그인
6. 데이터셋 로드
7. 데이터셋 포맷팅
8. MedGemma 로드
9. LoRA 설정
10. Google Drive 연결
11. 학습
12. 필요하면 체크포인트에서 재개
13. 추론 테스트
14. Hugging Face Hub 업로드

### 중요

패키지 설치 셀 실행 후에는 반드시:

`런타임 → 세션 다시 시작`

또는

`런타임 → 런타임 다시 시작`

을 실행하세요.

Colab에서 이미 로드된 NumPy와 새로 설치된 NumPy가 충돌하는 것을 방지하기 위한 과정입니다.


---

# 0. GPU 확인


In [ ]:
!nvidia-smi


정상적으로 T4 GPU가 연결되어 있다면 다음과 비슷한 정보가 출력됩니다.

```text
Tesla T4
```

GPU가 없다면 이후 MedGemma 학습을 진행하지 마세요.

---

# 1. 패키지 설치

## 중요 (numpy는 건드리지 않습니다)

한때 `numpy.dtype size changed, may indicate binary incompatibility` 에러 때문에
numpy를 특정 버전(1.26.4)으로 낮춰 고정했었는데, 이게 오히려 문제였습니다 — 지금 Colab
기본 이미지는 `opencv`, `jax`, `cupy`, `shap`, `cudf`, `tifffile`, `rasterio` 등
수십 개 사전 설치 패키지가 전부 `numpy>=2`를 요구하도록 이미 맞춰져 있어서, numpy를 2.0
밑으로 내리면 오히려 그 패키지들과 어긋나 같은 종류의 ABI 에러가 재발합니다.

그래서 **numpy 버전은 아예 지정하지 않고, Colab에 이미 깔린 걸 그대로 씁니다.** 우리가 필요한
패키지만 설치합니다.


In [ ]:
%pip install -q -U transformers accelerate peft bitsandbytes datasets huggingface_hub trl

# torchvision은 이 노트북(텍스트 전용 QLoRA 파인튜닝)에 필요 없음.
# Colab 기본 이미지의 torch/torchvision 버전이 서로 안 맞는 경우가 있어서
# (torch 2.13.0 vs torchvision이 요구하는 torch 2.11.0), 남겨두면 MedGemma처럼
# 멀티모달 모델 클래스를 로드할 때 torchvision::nms 관련 에러로 막힌다. 지워서 회피.
!pip uninstall -y -q torchvision


---

# 2. 반드시 런타임 재시작

## 이 셀은 설명용입니다.

패키지 설치가 완료되면 아래 메뉴를 직접 실행하세요.

```text
런타임
→ 런타임 다시 시작
```

또는 Colab UI에 따라:

```text
런타임
→ 세션 다시 시작
```

재시작 후 아래 셀부터 다시 실행합니다.

---

# 3. Python / NumPy / GPU 환경 확인


In [ ]:
import sys
import numpy
import torch

print("Python :", sys.version)
print("NumPy  :", numpy.__version__)
print("PyTorch:", torch.__version__)

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)


정상적인 예 (numpy 버전은 Colab이 준 값 그대로면 됨, 특정 값을 강제하지 않음):

```text
CUDA available: True
GPU: Tesla T4
```

---

# 4. 핵심 패키지 Import 테스트

앞서 겪었던 numpy ABI 충돌이 없는지 먼저 확인합니다. 여기서 에러가 나면 numpy를 건드리는
다른 셀/명령을 실행한 적이 있는지 먼저 의심하고, `런타임 다시 시작` 후 재시도하세요.


In [ ]:
import numpy
import pandas
import pyarrow
import scipy

print("NumPy   :", numpy.__version__)
print("Pandas  :", pandas.__version__)
print("PyArrow :", pyarrow.__version__)
print("SciPy   :", scipy.__version__)

from datasets import load_dataset

print("datasets import OK")


여기서:

```text
datasets import OK
```

가 출력되어야 합니다.

---

# 5. 패키지 충돌 검사


In [ ]:
!pip check


여기서 의존성 문제가 출력되면 내용을 확인하세요.

특히 다음과 같은 패키지에서 문제가 없어야 합니다.

```text
numpy
pandas
pyarrow
scipy
datasets
transformers
peft
trl
bitsandbytes
```

---

# 6. Hugging Face 로그인

MedGemma를 사용하기 전에 Hugging Face에서 MedGemma 접근 권한을 승인하고 Token을 준비해야 합니다.

Colab:

```text
왼쪽 메뉴
→ 열쇠 아이콘
→ Secrets
→ HF_TOKEN
```

으로 등록하세요.

코드에 Token을 직접 입력하지 않습니다.


In [ ]:
from huggingface_hub import login
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN이 없습니다. "
        "Colab 왼쪽 메뉴의 Secrets에 HF_TOKEN을 등록하세요."
    )

login(token=HF_TOKEN)

print("Hugging Face login OK")


# 8. 데이터셋 로드 — 확장 조합 (11개 소스)

메인 노트북(KorMedMCQA+GenMedGPT-5k-ko, 2개 소스)과 달리, 이 노트북은 아래 11개
데이터셋을 모두 섞어서 학습한다. 전부 비상업적(포폴/프로젝트) 목적이라 CC-BY-NC나
라이선스 미표기 데이터셋도 배제하지 않았다.

**지식(사실 기반 QA) — 7개**

1. `ChuGyouk/Asan-AMC-Healthinfo`
2. `ChuGyouk/MedQA` (한국어)
3. `ChuGyouk/KoMedInstruct-52k`
4. `snuh/ClinicalQA`
5. `ChuGyouk/AI_healthcare_QA`
6. `ChuGyouk/HealthSearchQA-ko`
7. `hcw0329/medical-korean-alpaca`

**추론 보강(풀이 과정 포함) — 3개**

8. `ChuGyouk/medical-o1-reasoning-SFT-Ko`
9. `ChuGyouk/ChainOfDiagnosis-Ko`
10. `ChuGyouk/MedQA-Evol-Korean`

**대화형(멀티턴) — 1개**

11. `squarelike/ko_medical_chat`

큰 데이터셋은 학습 시간/균형을 위해 소스당 최대 2,000개로 샘플링하고, 원래 규모가
작은 데이터셋(ClinicalQA, HealthSearchQA-ko, ko_medical_chat)은 있는 그대로 다 쓴다.
전부 `{"prompt": [...], "completion": [...]}` 포맷으로 변환한 뒤 `concatenate_datasets`로
하나의 `dataset`에 합친다.


In [ ]:
import re
from datasets import load_dataset, concatenate_datasets


def _normalize_ws(value):
    """공백/개행 정규화 — 모든 데이터셋 포맷 함수가 공용으로 쓰는 유틸리티."""
    if value is None:
        return ""
    return re.sub(r"\s+", " ", str(value)).strip()


In [ ]:
# 여러 출처(지식/추론보강/대화형 11개)를 섞어 학습시킬 때 톤/제약을 통일하기 위한 공통
# 지시문. "두개내과", "소화기내시경센터"처럼 실존하지 않는 진료과명을 답변에 넣는 문제를
# 줄이려고, 실제 존재하는 전문과목 이름만 쓰라고 명시해둔다. MedQA류 객관식 포맷에서만
# 쓰는 옵션 키(A~D)도 여기서 공용으로 정의한다.
OPTION_KEYS = ["A", "B", "C", "D"]

INSTRUCTION_PREFIX = (
    "당신은 신중하고 정확한 의료 지식을 갖춘 병의원 진료상담 챗봇입니다. 확정적인 진단이나 "
    "처방 대신 가능성과 권장 사항을 안내하고, 진료과 이름은 내과·외과·소아청소년과·산부인과·"
    "신경과·신경외과·정신건강의학과·정형외과·이비인후과·피부과·안과·비뇨의학과·응급의학과·"
    "가정의학과 등 실제 존재하는 전문과목 명칭만 사용하세요."
)

# 소스당 상한 — 큰 데이터셋(2만~5만개대)이 작은 데이터셋을 압도하지 않도록 균형을 맞춘다.
PER_SOURCE_CAP = 2000


In [ ]:
# 지식① Asan-AMC-Healthinfo — 서울아산병원 건강정보 기반 instruction 데이터
ASAN_ID = "ChuGyouk/Asan-AMC-Healthinfo"
asan = load_dataset(ASAN_ID, split="train")
print(asan)
print(asan[0])


def format_asan(example):
    q = _normalize_ws(example.get("instruction"))
    a = _normalize_ws(example.get("output"))
    if not q or not a:
        return {"prompt": [], "completion": []}
    return {
        "prompt": [{"role": "user", "content": f"{INSTRUCTION_PREFIX}\n\n{q}"}],
        "completion": [{"role": "assistant", "content": a}],
    }


asan = asan.shuffle(seed=42).select(range(min(PER_SOURCE_CAP, len(asan))))
asan = asan.map(format_asan, remove_columns=[c for c in asan.column_names if c not in ("prompt", "completion")])
asan = asan.filter(lambda ex: bool(ex["prompt"]) and bool(ex["completion"]))
print(f"Asan-AMC-Healthinfo 변환 후: {len(asan)}개")

dataset = asan  # 첫 데이터셋이므로 여기서 dataset을 초기화한다
print(f"현재 전체: {len(dataset)}개")


---

## 지식② ~ ⑦

나머지 지식(사실 기반 QA) 6개 소스. 각 셀은 로드 → 첫 샘플 출력(스키마 확인용) →
포맷 변환 → `dataset`에 누적(concatenate) 순서로 동일하게 진행한다.


In [ ]:
# 지식② MedQA (한국어) — 미국 의사면허시험(USMLE) 기반 객관식, A~D 4지선다
MEDQA_ID = "ChuGyouk/MedQA"
medqa = load_dataset(MEDQA_ID, split="train")
print(medqa)
print(medqa[0])
# 주의: 위 출력에서 언어를 구분하는 컬럼(예: language/lang)이 보이면, 아래에서
# 한국어만 남기도록 필터를 추가해야 한다 — 이미 한국어 전용이면 그대로 둔다.


def format_medqa(example):
    options_text = "\n".join(
        f"{key}. {_normalize_ws(example[key])}"
        for key in OPTION_KEYS
        if example.get(key) is not None and _normalize_ws(example[key])
    )
    question = _normalize_ws(example.get("question"))
    if not question or not options_text:
        return {"prompt": [], "completion": []}

    answer_text = _normalize_ws(example.get("answer"))
    try:
        letter = OPTION_KEYS[int(example.get("answer_idx"))]
    except (TypeError, ValueError, IndexError):
        letter = None

    model_turn = f"정답은 {letter}번, {answer_text}입니다." if letter else f"정답은 {answer_text}입니다."
    return {
        "prompt": [{"role": "user", "content": f"{INSTRUCTION_PREFIX}\n\n{question}\n\n선택지:\n{options_text}"}],
        "completion": [{"role": "assistant", "content": model_turn}],
    }


medqa = medqa.shuffle(seed=42).select(range(min(PER_SOURCE_CAP, len(medqa))))
medqa = medqa.map(format_medqa, remove_columns=[c for c in medqa.column_names if c not in ("prompt", "completion")])
medqa = medqa.filter(lambda ex: bool(ex["prompt"]) and bool(ex["completion"]))
print(f"MedQA 변환 후: {len(medqa)}개")

dataset = concatenate_datasets([dataset, medqa]).shuffle(seed=42)
print(f"현재 전체: {len(dataset)}개")


In [ ]:
# 지식③ KoMedInstruct-52k — instruction/input/output 형식의 한국어 의료 instruction 데이터
KOMEDINSTRUCT_ID = "ChuGyouk/KoMedInstruct-52k"
komedinstruct = load_dataset(KOMEDINSTRUCT_ID, split="train")
print(komedinstruct)
print(komedinstruct[0])


def format_komedinstruct(example):
    instr = _normalize_ws(example.get("instruction"))
    inp = _normalize_ws(example.get("input"))
    out = _normalize_ws(example.get("output"))
    if not instr or not out:
        return {"prompt": [], "completion": []}
    question = f"{instr}\n\n{inp}" if inp and inp != "<noinput>" else instr
    return {
        "prompt": [{"role": "user", "content": f"{INSTRUCTION_PREFIX}\n\n{question}"}],
        "completion": [{"role": "assistant", "content": out}],
    }


komedinstruct = komedinstruct.shuffle(seed=42).select(range(min(PER_SOURCE_CAP, len(komedinstruct))))
komedinstruct = komedinstruct.map(
    format_komedinstruct,
    remove_columns=[c for c in komedinstruct.column_names if c not in ("prompt", "completion")],
)
komedinstruct = komedinstruct.filter(lambda ex: bool(ex["prompt"]) and bool(ex["completion"]))
print(f"KoMedInstruct-52k 변환 후: {len(komedinstruct)}개")

dataset = concatenate_datasets([dataset, komedinstruct]).shuffle(seed=42)
print(f"현재 전체: {len(dataset)}개")


In [ ]:
# 지식④ snuh/ClinicalQA — 서울대병원 임상 QA (1,050개 규모라 샘플링 없이 전량 사용)
CLINICALQA_ID = "snuh/ClinicalQA"
clinicalqa = load_dataset(CLINICALQA_ID, split="train")
print(clinicalqa)
print(clinicalqa[0])
# 주의: answer/options의 정확한 타입(문자열/리스트/딕셔너리)을 위 출력으로 확인하고,
# 아래 포맷 함수가 실제 구조와 맞는지 검증할 것.


def format_clinicalqa(example):
    question = _normalize_ws(example.get("question"))
    answer = _normalize_ws(str(example.get("answer", "")))
    explanation = _normalize_ws(example.get("explanation"))
    if not question or not answer:
        return {"prompt": [], "completion": []}
    model_turn = f"정답은 {answer}입니다."
    if explanation:
        model_turn += f"\n\n해설: {explanation}"
    return {
        "prompt": [{"role": "user", "content": f"{INSTRUCTION_PREFIX}\n\n{question}"}],
        "completion": [{"role": "assistant", "content": model_turn}],
    }


clinicalqa = clinicalqa.map(
    format_clinicalqa,
    remove_columns=[c for c in clinicalqa.column_names if c not in ("prompt", "completion")],
)
clinicalqa = clinicalqa.filter(lambda ex: bool(ex["prompt"]) and bool(ex["completion"]))
print(f"snuh/ClinicalQA 변환 후: {len(clinicalqa)}개")

dataset = concatenate_datasets([dataset, clinicalqa]).shuffle(seed=42)
print(f"현재 전체: {len(dataset)}개")


In [ ]:
# 지식⑤ AI_healthcare_QA — 환자 질문 + GPT-4o 계열 답변 쌍
AIHEALTHCAREQA_ID = "ChuGyouk/AI_healthcare_QA"
ai_healthcare_qa = load_dataset(AIHEALTHCAREQA_ID, split="train")
print(ai_healthcare_qa)
print(ai_healthcare_qa[0])


def format_ai_healthcare_qa(example):
    q = _normalize_ws(example.get("question"))
    a = _normalize_ws(example.get("gpt4o"))  # gpt4o-mini보다 더 큰 모델 답변을 우선 사용
    if not q or not a:
        return {"prompt": [], "completion": []}
    return {
        "prompt": [{"role": "user", "content": f"{INSTRUCTION_PREFIX}\n\n{q}"}],
        "completion": [{"role": "assistant", "content": a}],
    }


ai_healthcare_qa = ai_healthcare_qa.shuffle(seed=42).select(range(min(PER_SOURCE_CAP, len(ai_healthcare_qa))))
ai_healthcare_qa = ai_healthcare_qa.map(
    format_ai_healthcare_qa,
    remove_columns=[c for c in ai_healthcare_qa.column_names if c not in ("prompt", "completion")],
)
ai_healthcare_qa = ai_healthcare_qa.filter(lambda ex: bool(ex["prompt"]) and bool(ex["completion"]))
print(f"AI_healthcare_QA 변환 후: {len(ai_healthcare_qa)}개")

dataset = concatenate_datasets([dataset, ai_healthcare_qa]).shuffle(seed=42)
print(f"현재 전체: {len(dataset)}개")


In [ ]:
# 지식⑥ HealthSearchQA-ko — 실제 건강 검색 질의 기반 QA (3,170개 규모, 전량 사용)
HEALTHSEARCHQA_ID = "ChuGyouk/HealthSearchQA-ko"
healthsearchqa = load_dataset(HEALTHSEARCHQA_ID, split="train")
print(healthsearchqa)
print(healthsearchqa[0])


def format_healthsearchqa(example):
    q = _normalize_ws(example.get("question_ko"))
    a = _normalize_ws(example.get("answer_ko"))
    if not q or not a:
        return {"prompt": [], "completion": []}
    return {
        "prompt": [{"role": "user", "content": f"{INSTRUCTION_PREFIX}\n\n{q}"}],
        "completion": [{"role": "assistant", "content": a}],
    }


healthsearchqa = healthsearchqa.map(
    format_healthsearchqa,
    remove_columns=[c for c in healthsearchqa.column_names if c not in ("prompt", "completion")],
)
healthsearchqa = healthsearchqa.filter(lambda ex: bool(ex["prompt"]) and bool(ex["completion"]))
print(f"HealthSearchQA-ko 변환 후: {len(healthsearchqa)}개")

dataset = concatenate_datasets([dataset, healthsearchqa]).shuffle(seed=42)
print(f"현재 전체: {len(dataset)}개")


In [ ]:
# 지식⑦ hcw0329/medical-korean-alpaca — alpaca 포맷 한국어 의료 instruction 데이터
MEDKO_ALPACA_ID = "hcw0329/medical-korean-alpaca"
medko_alpaca = load_dataset(MEDKO_ALPACA_ID, split="train")
print(medko_alpaca)
print(medko_alpaca[0])


def format_medko_alpaca(example):
    instr = _normalize_ws(example.get("instruction"))
    inp = _normalize_ws(example.get("input"))
    out = _normalize_ws(example.get("output"))
    if not instr or not out:
        return {"prompt": [], "completion": []}
    question = f"{instr}\n\n{inp}" if inp else instr
    return {
        "prompt": [{"role": "user", "content": f"{INSTRUCTION_PREFIX}\n\n{question}"}],
        "completion": [{"role": "assistant", "content": out}],
    }


medko_alpaca = medko_alpaca.shuffle(seed=42).select(range(min(PER_SOURCE_CAP, len(medko_alpaca))))
medko_alpaca = medko_alpaca.map(
    format_medko_alpaca,
    remove_columns=[c for c in medko_alpaca.column_names if c not in ("prompt", "completion")],
)
medko_alpaca = medko_alpaca.filter(lambda ex: bool(ex["prompt"]) and bool(ex["completion"]))
print(f"medical-korean-alpaca 변환 후: {len(medko_alpaca)}개")

dataset = concatenate_datasets([dataset, medko_alpaca]).shuffle(seed=42)
print(f"현재 전체: {len(dataset)}개")


---

## 추론 보강① ~ ③

정답만이 아니라 풀이 과정(chain-of-thought)이나 단계적 진단 흐름이 포함된 데이터.
답을 맞히는 능력보다 "왜 그런 결론에 도달했는지"를 자연스러운 문장으로 서술하는
능력을 보강하려는 목적이다.


In [ ]:
# 추론① medical-o1-reasoning-SFT-Ko — 질문 + 풀이과정(Complex_Cot) + 최종답변(Response)
MEDO1_ID = "ChuGyouk/medical-o1-reasoning-SFT-Ko"
med_o1 = load_dataset(MEDO1_ID, split="train")
print(med_o1)
print(med_o1[0])


def format_med_o1(example):
    q = _normalize_ws(example.get("Question"))
    cot = _normalize_ws(example.get("Complex_Cot"))
    resp = _normalize_ws(example.get("Response"))
    if not q or not resp:
        return {"prompt": [], "completion": []}
    model_turn = f"{cot}\n\n{resp}" if cot else resp
    return {
        "prompt": [{"role": "user", "content": f"{INSTRUCTION_PREFIX}\n\n{q}"}],
        "completion": [{"role": "assistant", "content": model_turn}],
    }


med_o1 = med_o1.shuffle(seed=42).select(range(min(PER_SOURCE_CAP, len(med_o1))))
med_o1 = med_o1.map(format_med_o1, remove_columns=[c for c in med_o1.column_names if c not in ("prompt", "completion")])
med_o1 = med_o1.filter(lambda ex: bool(ex["prompt"]) and bool(ex["completion"]))
print(f"medical-o1-reasoning-SFT-Ko 변환 후: {len(med_o1)}개")

dataset = concatenate_datasets([dataset, med_o1]).shuffle(seed=42)
print(f"현재 전체: {len(dataset)}개")


In [ ]:
# 추론② ChainOfDiagnosis-Ko — 환자-의사 다중턴 대화로 이어지는 단계적 진단 흐름
CHAINOFDX_ID = "ChuGyouk/ChainOfDiagnosis-Ko"
chain_of_dx = load_dataset(CHAINOFDX_ID, split="train")
print(chain_of_dx)
print(chain_of_dx[0])
# 주의: 정확한 대화 필드명을 위 출력으로 확인할 것 — 아래 _extract_dialogue_turns()가
# 후보 필드명 중 어느 것도 못 찾으면, 여기서 확인한 실제 필드명을 후보에 추가해야 한다.


def _extract_dialogue_turns(example):
    for key in ("conversations", "CoD_conversations", "dialogue", "conversation"):
        value = example.get(key)
        if value:
            return value
    return None


def format_chain_of_dx(example):
    turns = _extract_dialogue_turns(example)
    if not turns or len(turns) < 2:
        return {"prompt": [], "completion": []}

    def _role(turn):
        raw = str(turn.get("from") or turn.get("role") or "").lower()
        return "assistant" if raw in ("doctor", "assistant", "gpt") else "user"

    def _text(turn):
        return _normalize_ws(turn.get("value") or turn.get("content") or "")

    prompt_turns = [{"role": _role(t), "content": _text(t)} for t in turns[:-1] if _text(t)]
    last = turns[-1]
    if not prompt_turns or not _text(last):
        return {"prompt": [], "completion": []}
    prompt_turns[0]["content"] = f"{INSTRUCTION_PREFIX}\n\n{prompt_turns[0]['content']}"
    return {
        "prompt": prompt_turns,
        "completion": [{"role": "assistant", "content": _text(last)}],
    }


chain_of_dx = chain_of_dx.shuffle(seed=42).select(range(min(PER_SOURCE_CAP, len(chain_of_dx))))
chain_of_dx = chain_of_dx.map(
    format_chain_of_dx,
    remove_columns=[c for c in chain_of_dx.column_names if c not in ("prompt", "completion")],
)
chain_of_dx = chain_of_dx.filter(lambda ex: bool(ex["prompt"]) and bool(ex["completion"]))
print(f"ChainOfDiagnosis-Ko 변환 후: {len(chain_of_dx)}개")

dataset = concatenate_datasets([dataset, chain_of_dx]).shuffle(seed=42)
print(f"현재 전체: {len(dataset)}개")


In [ ]:
# 추론③ MedQA-Evol-Korean — MedQA를 Evol-Instruct 방식으로 복잡화한 한국어 버전
MEDQA_EVOL_ID = "ChuGyouk/MedQA-Evol-Korean"
medqa_evol = load_dataset(MEDQA_EVOL_ID, split="train")
print(medqa_evol)
print(medqa_evol[0])


def format_medqa_evol(example):
    q = _normalize_ws(example.get("input"))
    a = _normalize_ws(example.get("output"))
    if not q or not a:
        return {"prompt": [], "completion": []}
    return {
        "prompt": [{"role": "user", "content": f"{INSTRUCTION_PREFIX}\n\n{q}"}],
        "completion": [{"role": "assistant", "content": a}],
    }


medqa_evol = medqa_evol.shuffle(seed=42).select(range(min(PER_SOURCE_CAP, len(medqa_evol))))
medqa_evol = medqa_evol.map(
    format_medqa_evol,
    remove_columns=[c for c in medqa_evol.column_names if c not in ("prompt", "completion")],
)
medqa_evol = medqa_evol.filter(lambda ex: bool(ex["prompt"]) and bool(ex["completion"]))
print(f"MedQA-Evol-Korean 변환 후: {len(medqa_evol)}개")

dataset = concatenate_datasets([dataset, medqa_evol]).shuffle(seed=42)
print(f"현재 전체: {len(dataset)}개")


---

## 대화형①

선택지 없이 환자가 증상을 자유 서술하는 멀티턴 대화. 지식/추론보강 데이터가 모두
"단일 질문 → 단일 답변" 구조인 것과 달리, 실제 상담에 가까운 여러 턴짜리 대화 형태를
학습 데이터에 포함시켜 상담 톤/흐름을 보강한다.


In [ ]:
# 대화형① squarelike/ko_medical_chat — client/doctor 멀티턴 대화 (3,040개 규모, 전량 사용)
KOMEDCHAT_ID = "squarelike/ko_medical_chat"
ko_medical_chat = load_dataset(KOMEDCHAT_ID, split="train")
print(ko_medical_chat)
print(ko_medical_chat[0])


def format_ko_medical_chat(example):
    turns = example.get("conversations")
    if not turns or len(turns) < 2:
        return {"prompt": [], "completion": []}

    def _role(turn):
        return "assistant" if str(turn.get("from", "")).lower() == "doctor" else "user"

    def _text(turn):
        return _normalize_ws(turn.get("value", ""))

    prompt_turns = [{"role": _role(t), "content": _text(t)} for t in turns[:-1] if _text(t)]
    last = turns[-1]
    if not prompt_turns or _role(last) != "assistant" or not _text(last):
        return {"prompt": [], "completion": []}
    prompt_turns[0]["content"] = f"{INSTRUCTION_PREFIX}\n\n{prompt_turns[0]['content']}"
    return {
        "prompt": prompt_turns,
        "completion": [{"role": "assistant", "content": _text(last)}],
    }


ko_medical_chat = ko_medical_chat.map(
    format_ko_medical_chat,
    remove_columns=[c for c in ko_medical_chat.column_names if c not in ("prompt", "completion")],
)
ko_medical_chat = ko_medical_chat.filter(lambda ex: bool(ex["prompt"]) and bool(ex["completion"]))
print(f"ko_medical_chat 변환 후: {len(ko_medical_chat)}개")

dataset = concatenate_datasets([dataset, ko_medical_chat]).shuffle(seed=42)
print(f"\n=== 11개 소스 혼합 완료: 총 {len(dataset)}개 ===")


---

# 13-2. 합친 데이터셋 저장/공유 (팀원 비교용)

지금까지 만든 `dataset`(8번 섹션의 11개 소스를 전부 혼합하고 셔플까지 끝난 상태)은 이 Colab
세션 메모리에만 있고 어디에도 저장되지 않는다 — 세션이 끊기면 사라지고, 다시 만들어도 셔플
시드나 업스트림 데이터셋 변경 때문에 완전히 같다는 보장이 없다.

**여러 모델(medgemma/gemma/qwen/llama)을 공정하게 비교하려면 전부 같은 데이터 스냅샷으로
학습해야 한다** — 그래서 지금 이 시점의 `dataset`을 얼려서 HF Hub에 비공개로 올려둔다.
팀원이 합친 데이터셋을 비교하고 싶다고 했으면, 이 리포지토리 이름을 공유하면 된다.


In [ ]:
MIXED_DATASET_REPO = "gon-0130/medgemma-mixed-dataset-alt"

dataset.push_to_hub(MIXED_DATASET_REPO, private=True)

print(f"혼합 데이터셋 업로드 완료: https://huggingface.co/datasets/{MIXED_DATASET_REPO}")
print(f"불러올 때: load_dataset('{MIXED_DATASET_REPO}')")
print(
    "팀원과 비교할 땐 이 리포지토리 이름과 링크를 공유하면 됩니다 — private 리포라서 "
    "HF 계정에 접근 권한(Settings > Collaborators)을 따로 추가해줘야 팀원이 열어볼 수 있습니다."
)


예상 형태:

```python
{
    "prompt": [{"role": "user", "content": "문제 내용\n\n선택지:\nA. ...\n..."}],
    "completion": [{"role": "assistant", "content": "정답은 C번, ...입니다."}],
}
```

`<start_of_turn>...` 같은 특수토큰을 직접 안 써도, SFTTrainer가 이 conversational
prompt-completion 포맷을 보면 자동으로 `tokenizer.apply_chat_template()`을 적용해서
Gemma 고유 포맷으로 감싸준다. role은 trl이 받는 표준 이름인 `"user"`/`"assistant"`를 쓰고,
`"assistant"`는 Gemma의 chat_template 내부에서 자동으로 `<start_of_turn>model`로
바뀐다("model"을 직접 role로 넣으면 trl이 `ValueError`를 낸다).

---

# 14. MedGemma 모델 설정


In [ ]:
import torch

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

MODEL_ID = "google/medgemma-4b-it"


---

# 15. 4bit 양자화 설정

## T4에서는 float16을 사용합니다.

T4(Turing 아키텍처)는 bf16 텐서코어 가속을 지원하지 않습니다(Ampere 이상부터 지원). 그래서
T4 환경에서 안정적/효율적으로 쓰려면 BF16 대신 FP16을 씁니다.


In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)


---

# 16. Tokenizer 로드


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
)

tokenizer.padding_side = "right"

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


---

# 17. MedGemma 모델 로드


In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16,  # transformers 최신 버전은 torch_dtype 대신 dtype을 씀
    token=HF_TOKEN,
)


---

# 18. 모델 메모리 상태 확인


In [ ]:
print("Model loaded successfully")

if torch.cuda.is_available():
    print(
        "GPU memory allocated:",
        round(torch.cuda.memory_allocated() / 1024**3, 2),
        "GB"
    )

    print(
        "GPU memory reserved:",
        round(torch.cuda.memory_reserved() / 1024**3, 2),
        "GB"
    )


---

# 19. LoRA 설정


In [ ]:
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)


---

# 20. K-bit Training 준비


In [ ]:
model = prepare_model_for_kbit_training(model)

model.config.use_cache = False


---

# 21. LoRA Configuration


In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    ],
    bias="none",
    task_type="CAUSAL_LM",
)


---

# 22. LoRA 적용


In [ ]:
model = get_peft_model(
    model,
    lora_config,
)

model.print_trainable_parameters()


출력 결과에서 전체 파라미터 대비 trainable parameter가 매우 적게 나오는 것이 정상입니다.

---

# 23. Google Drive 연결

Colab 세션이 종료되면 `/content`에 저장된 파일이 사라질 수 있습니다.

따라서 학습 체크포인트를 Google Drive에 저장합니다.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

CKPT_DIR = (
    "/content/drive/MyDrive/"
    "medgemma-lora-ckpt"
)

print("Checkpoint directory:", CKPT_DIR)


---

# 24. 학습 설정

## T4 16GB 기준


In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir=CKPT_DIR,

    # T4 16GB
    per_device_train_batch_size=1,

    # 실제 batch size를 늘리는 효과
    gradient_accumulation_steps=8,

    # 1 epoch로는 정답이 짧고 정형화된 패턴(예: "정답은 C번, ...입니다")에 노출되는
    # 총 스텝 수가 적어서, 자유 서술형 질문에 그 패턴을 못 벗어나고 반복 생성에 빠지기 쉬웠다.
    # 정제된 데이터(10-1번)는 약 1,800개 정도라 2 epoch도 T4 무료 세션에서 충분히 감당된다.
    # 그래도 반복 증상이 남으면 3까지 올려보되, 과적합 신호(loss가 비정상적으로 빨리 0에
    # 가까워짐)가 보이면 다시 줄인다.
    num_train_epochs=2,

    learning_rate=2e-4,

    # T4는 bf16 텐서코어가 없어서 bf16=False. fp16=True(AMP GradScaler)는 Gemma 계열에서
    # LoRA 레이어 일부가 bfloat16으로 생성되는 경우가 있어 GradScaler가 그 텐서를 처리 못 해
    # "_amp_foreach_non_finite_check_and_unscale_cuda not implemented for BFloat16" 에러가 남.
    # 4bit 베이스 자체가 이미 압축돼있고 LoRA 파라미터는 작아서, AMP 없이 기본 정밀도로 학습.
    fp16=False,
    bf16=False,

    # 메모리 절약
    gradient_checkpointing=True,

    logging_steps=10,

    # Colab 세션 종료 대비
    save_steps=50,
    save_total_limit=3,

    # dataset이 이제 text 단일 필드가 아니라 prompt/completion으로 나뉜 conversational
    # 포맷이라(24-1번 참고) dataset_text_field는 더 안 씀. completion_only_loss는
    # prompt-completion 포맷에서 기본값이 True라 안 적어도 되지만, 명시적으로 남겨둔다.
    completion_only_loss=True,
    max_length=1024,

    # W&B 등의 외부 로깅 방지
    report_to="none",

    # 데이터 packing
    packing=False,
)


---

# 24-1. 정답 부분에만 Loss 집중시키기 (Completion-only Loss)

지금까지는 `text` 필드 전체(질문+선택지+정답)에 대해 언어모델 loss를 계산했다. 그런데 실제로
잘 배우길 원하는 건 "정답을 어떻게 생성하는가"이지, 매번 주어지는 질문/선택지 텍스트를 다시
예측하는 게 아니다. 질문 부분까지 loss에 포함되면 학습 신호가 흐려지고, 정답이 몇 단어로
끝나는 데이터에서는 정답 부분의 상대적 비중이 더 작아져서 반복 생성 경향을 키우는 요인이
될 수 있다.

**(업데이트) `DataCollatorForCompletionOnlyLM`은 최신 trl에서 제거됐다** — 대신
`SFTConfig(completion_only_loss=True)`를 쓰면 되는데, 이건 `text` 단일 필드가 아니라
**prompt/completion으로 나뉜 데이터셋**에서만 동작한다. 그래서 8번 섹션의 11개 데이터셋
포맷 함수(`format_asan`/`format_medqa`/... 등)가 전부 `{"prompt": [...], "completion": [...]}`
형태로 반환하도록 이미 만들어뒀다 — 이 셀은 별도 collator를 만드는 대신, 그 포맷이 제대로
됐는지만 확인한다.


In [ ]:
# prompt/completion 포맷이 제대로 됐는지, chat template이 실제로 적용되는지 확인한다.
sample = dataset[0]
assert "prompt" in dataset.column_names and "completion" in dataset.column_names, (
    "dataset에 prompt/completion 컬럼이 없다 — 8번 섹션의 포맷팅 함수들이 최신 버전으로 "
    "적용됐는지 확인(런타임을 재시작하지 않고 예전 셀 결과가 남아있는 경우 이럴 수 있음)."
)

rendered = tokenizer.apply_chat_template(
    sample["prompt"] + sample["completion"],
    tokenize=False,
)
print("apply_chat_template 렌더링 결과:\n")
print(rendered)


---

# 25. SFTTrainer 생성


In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
)


---

# 26. 학습 전 Trainer 확인


In [ ]:
print(trainer)


---

# 27. 학습 시작

## 처음에는 1 epoch로 테스트하는 것을 권장합니다.


In [ ]:
trainer.train()


학습이 정상적으로 시작되면 다음과 비슷한 로그가 출력됩니다.

```text
***** Running training *****
Num examples = ...
Num Epochs = 1
...
```

---

# 28. 학습 결과 저장

학습이 정상적으로 끝난 후 LoRA Adapter를 저장합니다.


In [ ]:
FINAL_ADAPTER_DIR = (
    "/content/drive/MyDrive/"
    "medgemma-lora-final"
)

trainer.save_model(FINAL_ADAPTER_DIR)

tokenizer.save_pretrained(
    FINAL_ADAPTER_DIR
)

print(
    "Adapter saved to:",
    FINAL_ADAPTER_DIR
)


---

# 29. 학습 체크포인트 확인


In [ ]:
import os

print(os.listdir(CKPT_DIR))


---

# 30. 세션이 끊긴 경우 체크포인트에서 재개

Colab 세션이 종료된 경우:

1. 런타임 재연결
2. 패키지 설치
3. 런타임 재시작
4. HF 로그인
5. 데이터셋 로드
6. 모델 로드
7. LoRA 설정
8. Drive mount
9. Trainer 생성

까지 다시 실행합니다.

그 다음:


In [ ]:
import os

checkpoint_dirs = [
    os.path.join(CKPT_DIR, name)
    for name in os.listdir(CKPT_DIR)
    if name.startswith("checkpoint-")
]

checkpoint_dirs = [
    path
    for path in checkpoint_dirs
    if os.path.isdir(path)
]

if checkpoint_dirs:
    checkpoint_dirs.sort(
        key=lambda path: int(
            os.path.basename(path).split("-")[-1]
        )
    )

    latest_checkpoint = checkpoint_dirs[-1]

    print(
        "Latest checkpoint:",
        latest_checkpoint
    )

    trainer.train(
        resume_from_checkpoint=latest_checkpoint
    )

else:
    print("체크포인트가 없습니다. 처음부터 학습을 시작하세요.")


---

# 31. 간단한 추론 테스트

학습이 완료되었으면 먼저 Colab 안에서 모델이 제대로 답변하는지 테스트합니다.


추론 전에 학습 때 꺼뒀던 캐시를 다시 켭니다 — 안 켜면 두통이 3일째 있어요 같은
질문에 같은 글자만 반복하는("두두두두...") 증상이 날 수 있습니다.


In [ ]:
model.eval()
model.config.use_cache = True
model.gradient_checkpointing_disable()


In [ ]:
prompt = (
    "<start_of_turn>user\n"
    "두통이 3일째 있어요. "
    "어떤 진료과를 방문하는 것이 좋을까요?"
    "<end_of_turn>\n"
    "<start_of_turn>model\n"
)

inputs = tokenizer(
    prompt,
    return_tensors="pt"
)

inputs = {
    key: value.to(model.device)
    for key, value in inputs.items()
}

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False,
    )

result = tokenizer.decode(
    output[0],
    skip_special_tokens=True,
)

print(result)


---

# 32. 다른 질문으로 테스트


In [ ]:
test_questions = [
    "3일째 열이 나고 기침이 있어요.",
    "가슴이 갑자기 아프고 숨쉬기가 힘들어요.",
    "복통이 계속되는데 어느 진료과에 가야 하나요?",
]

for question in test_questions:

    prompt = (
        "<start_of_turn>user\n"
        f"{question}"
        "<end_of_turn>\n"
        "<start_of_turn>model\n"
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    )

    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
        )

    result = tokenizer.decode(
        output[0],
        skip_special_tokens=True,
    )

    print("=" * 80)
    print("질문:", question)
    print(result)


---

# 32-1. 정답 정확도 검증 (held-out test set)

지금까지 31~32번은 사람이 답을 눈으로 훑어보는 것뿐이었다 — 이건 "그럴듯해 보인다"는 인상이지
검증이 아니다. 실제로 "정답에 근접한지"를 수치로 확인하려면 **학습에 한 번도 안 쓴 정답 세트**가
필요하다.

이 노트북은 KorMedMCQA를 쓰지 않으므로, 11개 소스 중 유일하게 A~D 선택지 객관식 구조를 가진
`ChuGyouk/MedQA`의 `test` split을 held-out 검증셋으로 쓴다 — 학습(8-지식②)에는 `train` split만
썼고 `test`는 지금까지 한 번도 안 건드렸으니 순수한 held-out이다. 여기서 모델이 생성한 답변에
**정답 선택지 글자(A~D) 또는 정답 텍스트**가 포함되는지로 채점한다.

**주의**: 이 방식은 "사실을 정확히 골랐는가"만 측정한다 — 어투/공감 표현/면책 문구 같은 상담
스타일은 이 지표로 안 잡힌다(그건 사람이 직접 몇 개 읽어보거나, 더 강한 모델로 채점하는
방식이 필요 — 별개 문제다). 전체를 다 돌리면 시간이 걸려서 우선 `EVAL_SAMPLE_N`개만 샘플링한다.


In [ ]:
import re as _re

eval_dataset = load_dataset(MEDQA_ID, split="test")
print(f"test split 크기: {len(eval_dataset)}개 (학습에 안 쓴 held-out)")

EVAL_SAMPLE_N = min(100, len(eval_dataset))  # 전부 돌리려면 len(eval_dataset)로 바꾸면 됨


def _eval_gold_answer(example):
    answer_text = _normalize_ws(example.get("answer"))
    try:
        letter = OPTION_KEYS[int(example.get("answer_idx"))]
    except (TypeError, ValueError, IndexError):
        letter = None
    return letter, answer_text


def _eval_prompt(example):
    options_text = "\n".join(
        f"{key}. {_normalize_ws(example[key])}"
        for key in OPTION_KEYS
        if example.get(key) is not None and _normalize_ws(example[key])
    )
    return (
        "<start_of_turn>user\n"
        f"{_normalize_ws(example['question'])}\n\n선택지:\n{options_text}"
        "<end_of_turn>\n<start_of_turn>model\n"
    )


correct = 0
wrong_examples = []

for example in eval_dataset.select(range(EVAL_SAMPLE_N)):
    prompt = _eval_prompt(example)
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=100, do_sample=False)
    response = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    gold_letter, gold_text = _eval_gold_answer(example)
    # 정답 글자를 답변 맨 앞부분에 언급했거나, 정답 텍스트 자체를 포함하면 정답으로 채점
    mentioned_letter = bool(gold_letter) and _re.search(rf"\b{gold_letter}\b", response[:20]) is not None
    mentioned_text = bool(gold_text) and gold_text in response
    is_correct = mentioned_letter or mentioned_text

    correct += int(is_correct)
    if not is_correct:
        wrong_examples.append({
            "question": example["question"][:60],
            "gold": f"{gold_letter}. {gold_text}",
            "response": response[:150],
        })

accuracy = correct / EVAL_SAMPLE_N
print(f"\n정확도: {correct}/{EVAL_SAMPLE_N} = {accuracy:.1%}")
print(f"\n틀린 예시 (최대 5개):")
for w in wrong_examples[:5]:
    print(f"- 질문: {w['question']}...")
    print(f"  정답: {w['gold']}")
    print(f"  모델 답변: {w['response']}...")
    print()


---

# 33. Hugging Face Hub에 LoRA Adapter 업로드

## 먼저 Hugging Face에서 업로드할 Repository를 준비하세요.

예:

```text
your-hf-account/medgemma-4b-lora-consultation
```

아래의 `<HF계정>`을 실제 계정명으로 변경합니다.


In [ ]:
ADAPTER_REPO = (
    "gon-0130/medgemma-4b-lora-consultation-alt"
)


---

# 34. Adapter 업로드


In [ ]:
model.push_to_hub(
    ADAPTER_REPO,
    private=True,
)

tokenizer.push_to_hub(
    ADAPTER_REPO,
    private=True,
)

print(
    "Uploaded to Hugging Face:",
    ADAPTER_REPO
)


---

# 35. 중요: GitHub와 Hugging Face의 역할

이 Notebook에서 GitHub는 학습 코드 관리용입니다.

```text
GitHub
├── train_medgemma_lora.ipynb
├── training code
└── configuration
```

학습된 모델 Adapter는 Hugging Face에 저장합니다.

```text
Hugging Face
└── medgemma-4b-lora-consultation
    ├── adapter_config.json
    ├── adapter_model.safetensors
    └── tokenizer files
```

GitHub에 대용량 모델 파일을 직접 올리지 않습니다.

---

# 36. 이후 FastAPI에서 사용하는 구조

학습이 끝난 후에는:

```text
MedGemma Base Model
        +
LoRA Adapter
        ↓
Fine-tuned MedGemma
        ↓
FastAPI
        ↓
POST /api/v1/ai/chat
```

구조로 사용할 수 있습니다.

**아래는 코드가 아니라 설명용 스니펫입니다 — 이 셀은 실행하지 마세요.** `base_model = ...`은
자리표시일 뿐 실제로 동작하는 코드가 아니고, 이건 이 학습 노트북이 아니라 나중에
`ai/llm`(현재 빈 패키지)에 서버 쪽 로딩 코드를 짤 때 참고할 형태입니다:

```python
from peft import PeftModel

base_model = ...  # 서버 쪽에서 medgemma-4b-it을 로드한 것

model = PeftModel.from_pretrained(
    base_model,
    "<HF계정>/medgemma-4b-lora-consultation"
)
```


---

# 37. 학습 데이터에 대한 주의사항

이 노트북은 11개 데이터셋(지식 7개 + 추론보강 3개 + 대화형 1개, 8번 섹션 참고)을 섞어서
학습한다. 라이선스가 명시되지 않았거나 CC-BY-NC(비영리) 조건인 소스가 섞여 있는데, 이
프로젝트는 상용화하지 않는 포트폴리오/프로젝트 목적이라 배제하지 않았다.

실제 상용 서비스로 발전시키는 경우에는 각 데이터셋 라이선스와 모델 라이선스를 다시 개별
검토해야 한다. 또한 실제 환자 개인정보가 포함된 데이터는 GitHub나 공개 Hugging Face
Repository에 업로드하지 마세요.

---

# 38. 최종 전체 흐름

```text
Google Colab
    │
    ├── T4 GPU
    │
    ├── NumPy (Colab 기본값, 강제로 안 낮춤)
    │
    ├── 11개 데이터셋 (지식 7 + 추론보강 3 + 대화형 1)
    │
    ├── MedGemma 4B
    │
    ├── 4bit Quantization
    │
    └── LoRA
            │
            ▼
       Fine-tuning
            │
            ▼
       Google Drive
       Checkpoint
            │
            ▼
       LoRA Adapter
            │
            ▼
      Hugging Face Hub
            │
            ▼
       FastAPI / Cloud Run
            │
            ▼
       실제 AI 서비스
```

# 핵심 체크리스트

* [ ] Colab GPU가 T4로 연결되어 있는가?
* [ ] numpy 버전을 억지로 낮추지 않고 Colab 기본값을 그대로 쓰고 있는가?
* [ ] 패키지 설치 후 런타임을 재시작했는가?
* [ ] `from datasets import load_dataset`가 정상 실행되는가?
* [ ] `pip check`에서 심각한 dependency conflict가 없는가?
* [ ] Hugging Face `HF_TOKEN`이 Colab Secrets에 등록되어 있는가?
* [ ] MedGemma 접근 권한이 있는가?
* [ ] GitHub clone은 현재 단계에서 생략했는가?
* [ ] Google Drive가 연결되어 있는가?
* [ ] T4에서 `fp16=False`, `bf16=False`로 설정했는가? (fp16 AMP는 Gemma 계열과 GradScaler 충돌 있음)
* [ ] 학습 전 Base Model 평가를 준비했는가?
* [ ] 학습 후 LoRA Adapter를 저장했는가?
* [ ] Hugging Face Repository를 private으로 설정했는가?
* [ ] 실제 서비스에서는 FastAPI가 Adapter를 불러오도록 구성했는가?


---

# 39. (선택) 재학습 없이 어댑터만 불러와서 빠르게 테스트

**세션이 끊겨서 다시 들어왔거나, 학습은 이미 끝나서 어댑터가 HF Hub에 올라가 있는 상태라면
이 섹션부터 실행하면 됩니다.** 데이터셋 로드/포맷팅(4~5번)이나 학습(9번) 전체를 다시 안 해도
됩니다 — 아래 5개 셀만 순서대로 실행하면 곧바로 테스트할 수 있습니다.

1. 패키지 설치(1번) → **런타임 다시 시작** → HF 로그인(2번)까지는 그대로 필요
2. 그다음 아래 셀들만 실행:


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

MODEL_ID = "google/medgemma-4b-it"
# 이 노트북(alt, 확장 데이터셋 실험)의 최신 어댑터. -alt 접미사로 v1/v2(스크리닝)/main과
# 구분한다.
ADAPTER_REPO = "gon-0130/medgemma-4b-lora-consultation-alt"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16,
    token=HF_TOKEN,
)

model = PeftModel.from_pretrained(base_model, ADAPTER_REPO)
model.eval()
model.config.use_cache = True

print("어댑터 로드 완료")


## 39-1. 반복 증상("두두두...", "정정정...") 잡기 위한 디코딩 옵션

`use_cache`를 켜도 같은 증상이면, 캐시 문제가 아니라 **탐욕적(greedy) 디코딩이 학습으로 좁아진
확률분포에 갇혀서** 그럴 가능성이 큽니다(1 epoch·짧고 반복적인 학습 타깃 특성상 흔함).
`repetition_penalty`와 `no_repeat_ngram_size`로 같은 토큰 반복을 명시적으로 막고,
`do_sample=True`로 약간의 무작위성을 줘서 탈출 가능성을 높입니다.


In [ ]:
def ask(question):
    prompt = (
        "<start_of_turn>user\n"
        f"{question}"
        "<end_of_turn>\n"
        "<start_of_turn>model\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.3,
            no_repeat_ngram_size=3,
        )
    return tokenizer.decode(output[0], skip_special_tokens=True)


print(ask("두통이 3일째 있어요. 어떤 진료과를 방문하는 것이 좋을까요?"))


**이래도 여전히 반복되면** 다음을 순서대로 시도한다.

1. `num_train_epochs`를 3으로 올려서 다시 학습 (단, loss가 비정상적으로 빨리 0에
   가까워지면 과적합이니 다시 낮춘다)
2. `PER_SOURCE_CAP`(8번 섹션)을 낮춰서 객관식(MedQA 등) 비중을 줄이고, 대화형/추론보강
   비중을 상대적으로 높여본다 — 이 노트북은 이미 대화형·추론보강 데이터를 섞고 있지만,
   그래도 객관식 정답 패턴이 남아있다면 그 비중을 더 낮추는 방향으로 조정한다.
3. 32-1번(held-out 정확도)과 여기 39-1번(자유 서술형 테스트)을 둘 다 확인해서, 어느 쪽이
   무너지는지로 원인을 좁힌다 — 객관식 정확도만 낮으면 학습 데이터/에폭 문제고, 자유
   서술형만 반복되면 디코딩 설정(위 셀의 `repetition_penalty`/`no_repeat_ngram_size`) 쪽을
   더 조정해본다.
